# Task 9 — Texture Robustness Stage 1

Thin Colab launcher only. All transform, model, inference, metric, and gate logic lives in `src/cya_detector/evaluation/texture_robustness.py`; this notebook invokes it only through `scripts/materialize_texture_robustness.py`, `scripts/evaluate_texture_robustness.py`, and `scripts/compare_texture_robustness.py` (the same CLIs wired to the `task9-robustness-*` Make targets).

**Frozen-checkpoint boundary.** This continuation trains nothing. It evaluates the nine existing immutable best-clean Task 9 checkpoints (`global_only`/`local_only`/`global_local` x seeds 42/43/44) over exactly nine texture-sensitive Task 3 cells materialized directly from the fixed-Q96 matched-clean `selection_val` parents: `jpeg_q90`/`jpeg_q70`/`jpeg_q50`/`jpeg_q30`, `blur_sigma_0.5`/`blur_sigma_1.0`/`blur_sigma_2.0`, and `resize_scale_0.5`/`resize_scale_0.25`. It never reads `seed_train`, `self_train_pool`, or sealed `final_test`.

**Controlled-RINE comparator.** The comparison restores and hash-verifies the three retained per-seed controlled-RINE `best_50_50.pt`/`best_50_50_predictions.csv` artifacts and recomputes their nine-cell subset from persisted per-sample predictions — it never substitutes the retained 14-cell `0.9981` aggregate.

**Decision gate.** `global_local` must beat both `global_only` and controlled RINE on the locked 50/50 score, aggregate class tolerance, and every per-cell worst-case condition (see `docs/superpowers/specs/2026-08-31-task-9-texture-robustness-stage1-design.md`). A `retain_texture_for_full_robustness` decision authorizes only a later, separately controlled evaluation of the remaining Task 3 noise/color-jitter/crop cells; it does not by itself retain Task 9, change calibration, or authorize `final_test`.

In [ ]:
# 1. GPU and Drive.
import torch
assert torch.cuda.is_available(), 'Task 9 Stage 1 requires a GPU Colab server'
print('GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Refresh repository and dependencies.
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

PROJECT_ROOT = Path('/content/cya-techjam26')
REPOSITORY_URL = 'https://github.com/maxi-cmyk/cya-techjam26.git'
if (PROJECT_ROOT / '.git').is_dir():
    status = subprocess.run(['git', 'status', '--porcelain'], cwd=PROJECT_ROOT, check=True, capture_output=True, text=True).stdout.splitlines()
    unexpected = [line for line in status if not line.endswith('configs/colab.json')]
    assert not unexpected, f'Unexpected remote checkout changes: {unexpected}'
    if status:
        subprocess.run(['git', 'restore', 'configs/colab.json'], cwd=PROJECT_ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only'], cwd=PROJECT_ROOT, check=True)
else:
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
subprocess.run(['git', 'checkout', 'task9-texture-robustness-stage1'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=PROJECT_ROOT, check=True)

In [ ]:
# 3. Pin the exact model commit and print the frozen Stage-1 contract.
from huggingface_hub import model_info
config_path = PROJECT_ROOT / 'configs/colab.json'
config = json.loads(config_path.read_text())
resolved_commit = model_info(config['model']['identifier'], revision=config['model']['revision']).sha
config['model']['revision'] = resolved_commit
config_path.write_text(json.dumps(config, indent=2) + '\n')
print('Commit:', resolved_commit)
print('Stage-1 contract:', json.dumps(config['texture_robustness_stage1'], indent=2))

In [ ]:
# 4. Stage the fixed-Q96 manifest and matched-clean images locally.
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts'
DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/cya-techjam26/artifacts')
TASK2_ROOT = ARTIFACT_ROOT / 'task2'
input_archive = DRIVE_ARTIFACT_ROOT / 'task2_stagea_bundle.tar.gz'
assert input_archive.is_file(), input_archive
TASK2_ROOT.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(input_archive, TASK2_ROOT)
manifest = TASK2_ROOT / 'fixed_q96_manifest.csv'
assert manifest.is_file(), manifest
print('Task 9 manifest ready:', manifest)

In [ ]:
# 5. Restore the nine completed clean Task 9 runs and the three retained
#    controlled-RINE seed bundles from Drive. Both are read-only inputs to
#    Stage 1: nothing here trains or mutates either artifact set.
CLEAN_ROOT = ARTIFACT_ROOT / 'task9' / 'clean_pilot_v1'
DRIVE_CLEAN_ROOT = DRIVE_ARTIFACT_ROOT / 'task9' / 'clean_pilot_v1'
CONTROLLED_RINE_ROOT = ARTIFACT_ROOT / 'robustness' / 'train-controlled-rine'
DRIVE_CONTROLLED_RINE_ROOT = DRIVE_ARTIFACT_ROOT / 'robustness' / 'train-controlled-rine'

assert DRIVE_CLEAN_ROOT.is_dir(), f'Nine completed clean Task 9 runs not found on Drive: {DRIVE_CLEAN_ROOT}'
assert DRIVE_CONTROLLED_RINE_ROOT.is_dir(), f'Controlled-RINE artifacts not found on Drive: {DRIVE_CONTROLLED_RINE_ROOT}'
shutil.copytree(DRIVE_CLEAN_ROOT, CLEAN_ROOT, dirs_exist_ok=True)
shutil.copytree(DRIVE_CONTROLLED_RINE_ROOT, CONTROLLED_RINE_ROOT, dirs_exist_ok=True)
for variant in config['texture']['variants']:
    for seed in config['texture']['seeds']:
        checkpoint = CLEAN_ROOT / variant / f'seed_{seed}' / 'checkpoints' / 'best_clean.pt'
        assert checkpoint.is_file(), checkpoint
for seed in config['texture']['seeds']:
    predictions = CONTROLLED_RINE_ROOT / f'seed_{seed}' / 'best_50_50_predictions.csv'
    assert predictions.is_file(), predictions
print('Restored nine clean runs and three controlled-RINE seed bundles.')

In [ ]:
# 6. Materialize the nine locked Stage-1 cells directly from the fixed-Q96
#    clean parents. Large transformed images stay under /content; only the
#    manifest and report are durable, and both are copied to Drive after
#    this cell succeeds so a fresh runtime can resume without rematerializing.
STAGE1_IMAGE_ROOT = Path('/content/task9_robustness_stage1/images')
STAGE1_OUTPUT_ROOT = ARTIFACT_ROOT / 'task9' / 'clean_pilot_v1' / 'robustness_stage1_v1'
DRIVE_STAGE1_ROOT = DRIVE_ARTIFACT_ROOT / 'task9' / 'clean_pilot_v1' / 'robustness_stage1_v1'
STAGE1_MANIFEST = STAGE1_OUTPUT_ROOT / 'materialization' / 'transformed_selection_val.csv'
STAGE1_REPORT = STAGE1_OUTPUT_ROOT / 'materialization' / 'materialization_report.json'

drive_materialization = DRIVE_STAGE1_ROOT / 'materialization'
if drive_materialization.is_dir() and not STAGE1_MANIFEST.is_file():
    shutil.copytree(drive_materialization, STAGE1_OUTPUT_ROOT / 'materialization', dirs_exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/materialize_texture_robustness.py',
    '--input-manifest', str(manifest),
    '--output-root', str(STAGE1_IMAGE_ROOT),
    '--output-manifest', str(STAGE1_MANIFEST),
    '--report', str(STAGE1_REPORT),
], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

DRIVE_STAGE1_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(STAGE1_OUTPUT_ROOT / 'materialization', drive_materialization, dirs_exist_ok=True)
print('Stage-1 materialization ready:', STAGE1_MANIFEST)

In [ ]:
# 7. Extract transformed features once and evaluate all 81 frozen
#    variant/seed/cell prediction slices. Resumable: hash-verified completed
#    slices already restored from Drive are skipped; the feature cache stays
#    under /content and is never synced.
STAGE1_CACHE_ROOT = Path('/content/task9_robustness_stage1/cache')
STAGE1_PREDICTIONS_ROOT = STAGE1_OUTPUT_ROOT / 'predictions'
drive_predictions = DRIVE_STAGE1_ROOT / 'predictions'
if drive_predictions.is_dir():
    shutil.copytree(drive_predictions, STAGE1_PREDICTIONS_ROOT, dirs_exist_ok=True)

subprocess.run([
    sys.executable, 'scripts/evaluate_texture_robustness.py',
    '--transformed-manifest', str(STAGE1_MANIFEST),
    '--materialization-report', str(STAGE1_REPORT),
    '--clean-experiment-root', str(CLEAN_ROOT),
    '--cache-root', str(STAGE1_CACHE_ROOT),
    '--output-root', str(STAGE1_PREDICTIONS_ROOT),
    '--device', 'cuda',
], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

DRIVE_STAGE1_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(STAGE1_PREDICTIONS_ROOT, drive_predictions, dirs_exist_ok=True)
print('All 81 Stage-1 prediction slices ready:', STAGE1_PREDICTIONS_ROOT)

In [ ]:
# 8. Verify controlled RINE, recompute its nine-cell subset, and apply the
#    locked Stage-1 gate only after all 81 texture slices and all 27
#    controlled-RINE seed-cell partitions validate. The completion marker
#    `metadata/artifact_manifest.json` is copied to Drive last.
subprocess.run([
    sys.executable, 'scripts/compare_texture_robustness.py',
    '--clean-experiment-root', str(CLEAN_ROOT),
    '--robustness-root', str(STAGE1_PREDICTIONS_ROOT),
    '--controlled-rine-root', str(CONTROLLED_RINE_ROOT),
    '--output-root', str(STAGE1_OUTPUT_ROOT),
], cwd=PROJECT_ROOT, check=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})

shutil.copytree(STAGE1_OUTPUT_ROOT / 'reports', DRIVE_STAGE1_ROOT / 'reports', dirs_exist_ok=True)
shutil.copytree(STAGE1_OUTPUT_ROOT / 'metadata', DRIVE_STAGE1_ROOT / 'metadata', dirs_exist_ok=True)
comparison = json.loads((STAGE1_OUTPUT_ROOT / 'reports' / 'robustness_comparison.json').read_text())
print('Decision:', comparison['decision'])
print(json.dumps(comparison, indent=2))

## Reading the decision

- `retain_texture_for_full_robustness`: `global_local` beat both `global_only` and controlled RINE on the locked 50/50 score, held the aggregate class tolerance, and held every per-cell worst-case condition against both comparators. This authorizes a separately controlled evaluation of the remaining Task 3 noise/color-jitter/crop cells; it does not retain Task 9, change calibration, or authorize `final_test`.
- `reject_texture_robustness_stage1`: stop here. Controlled RINE remains the retained parent without the texture addition.

A `TextureRobustnessPrerequisiteError` raised by cell 8 means the retained controlled-RINE artifacts could not be verified (missing, incomplete, or hash-mismatched) — this blocks the comparison outright and is never itself a pass or rejection. Re-run cell 5 after confirming the Drive copy is complete.